# Notebook 02: Advanced Domain Feature Engineering Pipeline
**GuidedGuard – Explainable AI for Scam-Guided Digital Payment Detection**

--- 
### Feature Engineering Objectives:
Design and execute domain-specific feature engineering pipelines on clean processed datasets (`paysim_clean.csv` and `baf_clean.csv`) to generate 10 categories of fraud signals:
1. **Transaction Features**: Log amount, relative amount, percentile, amount deviation, large transaction flag.
2. **Temporal Features**: Hour of day, weekend flag, business hours flag, late night flag.
3. **Velocity Features**: 1h/6h/24h transaction velocity proxies, daily average, recent transaction spike.
4. **Balance Features**: Origin/destination balance errors, remaining balance %, balance wipeout flag (`balance_wipeout_orig`).
5. **Recipient Features**: Beneficiary transaction count, new beneficiary flag.
6. **Device Features**: Device usage count, device risk score, device change flag.
7. **Location Features**: Location change flag, high risk region flag, distance from home.
8. **Customer Behavior Features**: Average spending, spending deviation, spending pattern score.
9. **Risk Features**: Amount risk score, time risk score, recipient risk score, composite behavior risk score.
10. **Interaction Features**: Multiplicative terms (Amount × Velocity, Amount × Device Risk, Velocity × Recipient Risk, Time × Amount).

Clean feature datasets will be exported to:
- `data/processed/paysim_featured.csv`
- `data/processed/baf_featured.csv`

> **Strict Boundary Guardrails:**
> - ❌ No machine learning model training.
> - ❌ No train/test splitting.
> - ❌ No SHAP/LIME explainer initialization.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))
import config
from utils.data_loader import load_dataset
from preprocessing.preprocessing import save_processed_dataset
from preprocessing.feature_engineering import (
    create_transaction_features,
    create_temporal_features,
    create_velocity_features,
    create_balance_features,
    create_recipient_features,
    create_device_features,
    create_location_features,
    create_customer_features,
    create_risk_features,
    create_interaction_features,
    validate_engineered_features,
    feature_engineering_pipeline
)

print("Feature engineering environment initialized.")

# 1. Feature Engineering: PaySim Mobile Money Dataset
Generating domain features on `data/processed/paysim_clean.csv`.

In [ ]:
# Load clean PaySim dataset
paysim_clean_path = config.PROCESSED_DATA_DIR / "paysim_clean.csv"
paysim_clean = load_dataset(paysim_clean_path)
print(f"Loaded PaySim Clean Shape: {paysim_clean.shape}")

# Execute feature engineering pipeline
paysim_feat, paysim_feat_report = feature_engineering_pipeline(paysim_clean, dataset_name="PaySim")
print(f"PaySim Featured Shape: {paysim_feat.shape}")
print("Feature Validation Report:")
display(pd.Series(paysim_feat_report))

### PaySim Feature Inspection & Save (`data/processed/paysim_featured.csv`)

In [ ]:
# Save featured PaySim dataset
paysim_featured_path = config.RAW_DATA_DIR.parent / "processed" / "paysim_featured.csv"
save_processed_dataset(paysim_feat, paysim_featured_path)

print("Top 5 Rows of Engineered PaySim Features:")
display(paysim_feat.head())

# 2. Feature Engineering: Bank Account Fraud (BAF) Dataset
Generating domain features on `data/processed/baf_clean.csv`.

In [ ]:
# Load clean BAF dataset
baf_clean_path = config.PROCESSED_DATA_DIR / "baf_clean.csv"
baf_clean = load_dataset(baf_clean_path)
print(f"Loaded BAF Clean Shape: {baf_clean.shape}")

# Execute feature engineering pipeline
baf_feat, baf_feat_report = feature_engineering_pipeline(baf_clean, dataset_name="Bank Account Fraud")
print(f"BAF Featured Shape: {baf_feat.shape}")
print("Feature Validation Report:")
display(pd.Series(baf_feat_report))

### BAF Feature Inspection & Save (`data/processed/baf_featured.csv`)

In [ ]:
# Save featured BAF dataset
baf_featured_path = config.RAW_DATA_DIR.parent / "processed" / "baf_featured.csv"
save_processed_dataset(baf_feat, baf_featured_path)

print("Top 5 Rows of Engineered BAF Features:")
display(baf_feat.head())

# 3. Distribution Visualizations of Key Risk Features
Plotting distribution of composite risk scores across fraud vs legitimate classes.

In [ ]:
# PaySim: Composite Behavior Risk Score by Fraud Class
fig_ps_risk = px.histogram(
    paysim_feat,
    x="composite_behavior_risk_score",
    color="isFraud",
    barmode="overlay",
    title="PaySim: Composite Behavior Risk Score Distribution by Fraud Class",
    labels={"composite_behavior_risk_score": "Risk Score (0.0 - 1.0)", "isFraud": "Fraud (0=No, 1=Yes)"}
)
fig_ps_risk.show()

# BAF: Composite Behavior Risk Score by Fraud Class
fig_baf_risk = px.histogram(
    baf_feat,
    x="composite_behavior_risk_score",
    color="fraud_bool",
    barmode="overlay",
    title="BAF: Composite Behavior Risk Score Distribution by Fraud Class",
    labels={"composite_behavior_risk_score": "Risk Score (0.0 - 1.0)", "fraud_bool": "Fraud (0=No, 1=Yes)"}
)
fig_baf_risk.show()

# 4. Summary Comparison: Preprocessed vs Featured Datasets

In [ ]:
summary_comparison = pd.DataFrame([
    {
        "Dataset": "PaySim Mobile Money",
        "Clean Processed Shape": f"{paysim_clean.shape[0]} rows, {paysim_clean.shape[1]} cols",
        "Engineered Featured Shape": f"{paysim_feat.shape[0]} rows, {paysim_feat.shape[1]} cols",
        "New Features Added": paysim_feat.shape[1] - paysim_clean.shape[1],
        "Null Count": paysim_feat.isnull().sum().sum(),
        "Output Featured Path": str(paysim_featured_path)
    },
    {
        "Dataset": "Bank Account Fraud (BAF)",
        "Clean Processed Shape": f"{baf_clean.shape[0]} rows, {baf_clean.shape[1]} cols",
        "Engineered Featured Shape": f"{baf_feat.shape[0]} rows, {baf_feat.shape[1]} cols",
        "New Features Added": baf_feat.shape[1] - baf_clean.shape[1],
        "Null Count": baf_feat.isnull().sum().sum(),
        "Output Featured Path": str(baf_featured_path)
    }
])

display(summary_comparison)